# Notebook 06 — Diagnostic de séparabilité (Winter 2019)

**Question scientifique :** les vaches cliniquement boiteuses (score SLS) se distinguent-elles des vaches saines
à partir des données IceTag de McGill ?

**Pourquoi ce notebook :** l'application du pipeline du mémoire sur McGill ne reproduit pas les scores cliniques.
Avant de conclure quoi que ce soit sur le pipeline ou le mémoire, il faut répondre à une question préalable :
**le signal de boiterie est-il seulement présent dans les données IceTag de cette cohorte ?**

- Si AUCUNE feature ne sépare boiteuses/saines → le signal est absent des données. Aucun pipeline ne peut le détecter.
  Cela exonère à la fois le pipeline et le mémoire (c'est un problème de cohorte/données).
- Si certaines features séparent → le pipeline pourrait être recalibré sur McGill.

**Choix de Winter 2019 :** c'est le seul corpus dont les scores SLS (mars 2019) sont SYNCHRONES avec les
données IceTag (mars 2019). Fall 2019 a un décalage de 8 mois entre labels et capteurs → non concluant.

**Ce notebook ne modifie PAS le mémoire.** Il lit uniquement des données et produit un diagnostic.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, spearmanr
import warnings
warnings.filterwarnings('ignore')

# Racine du projet McGill (ce notebook est dans mcgill_iot_cattle/notebooks/)
PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_ROOT = PROJECT / 'Données completes' / 'Données accelerometres'
REPORTS = PROJECT / 'reports' / 'objective1_pipeline_icetag'
OUT = REPORTS / 'diagnostic_separabilite'
OUT.mkdir(parents=True, exist_ok=True)

ICETAG_INPUT = REPORTS / 'winter_2019_pipeline_input_15min.csv'
PIPELINE_SUMMARY = REPORTS / 'winter_2019_pipeline_summary.csv'
SLS_FILE = (DATA_ROOT / 'Winter 2019' / 'Icetag' / 'IceTags_Data' /
            'IceTags-issues and reports' / 'Exercise Study - SLS Scores.xlsx')

for p in [ICETAG_INPUT, PIPELINE_SUMMARY, SLS_FILE]:
    print('OK ' if p.exists() else 'MANQUANT ', p.name)

OK  winter_2019_pipeline_input_15min.csv
OK  winter_2019_pipeline_summary.csv
OK  Exercise Study - SLS Scores.xlsx


## 1. Chargement des scores cliniques SLS (labels de référence)

Le score SLS total = somme de 4 composantes (`Edge`, `Rest`, `Shiftwt`, `Uneven`).
`Shiftwt` (report de poids) et `Uneven` (démarche inégale) sont les signes cliniques de boiterie.
On définit :
- **boiteuse** = SLS total >= 2 (signes nets)
- **saine/légère** = SLS total < 2

In [2]:
sls = pd.read_excel(SLS_FILE)
sls = sls.rename(columns={'Unnamed: 6': 'SLS_total'})
sls['Cow'] = sls['Cow'].astype(str)
# Certaines vaches ont 2 lignes (2 vidéos) -> on prend le score max (pire cas)
sls_agg = sls.groupby('Cow')['SLS_total'].max().reset_index()
sls_agg['boiteuse'] = (sls_agg['SLS_total'] >= 2).astype(int)

print(f"Vaches scorées SLS : {len(sls_agg)}")
print(f"  boiteuses (SLS>=2) : {sls_agg['boiteuse'].sum()}")
print(f"  saines/légères     : {(sls_agg['boiteuse']==0).sum()}")
print(sls_agg.sort_values('SLS_total', ascending=False).to_string(index=False))

Vaches scorées SLS : 33
  boiteuses (SLS>=2) : 13
  saines/légères     : 20
 Cow  SLS_total  boiteuse
2047          2         1
3443          2         1
8513          2         1
 821          2         1
5874          2         1
5857          2         1
5295          2         1
5246          2         1
5221          2         1
3444          2         1
5259          2         1
2076          2         1
2069          2         1
2079          1         0
5250          1         0
2057          1         0
2056          1         0
8510          1         0
2081          1         0
2063          1         0
3437          0         0
8505          0         0
8512          0         0
8508          0         0
8506          0         0
5879          0         0
8502          0         0
8500          0         0
5875          0         0
5304          0         0
2078          0         0
5258          0         0
8514          0         0


## 2. Construction des features IceTag par vache

On calcule, pour chaque vache, des indicateurs au niveau journalier puis on en prend la moyenne :
- **Pas/jour** (réduction d'activité = signe possible de boiterie)
- **Heures couché/jour** (les vaches boiteuses se couchent souvent plus)
- **Nombre de couchers/jour** (lying bouts — indicateur classique)
- **Motion Index/jour** (intensité d'activité globale)
- **Écart-type des pas** (régularité de l'activité)

In [3]:
inp = pd.read_csv(ICETAG_INPUT)
inp['Cow'] = inp['Cow'].astype(str)
inp['Start'] = pd.to_datetime(inp['Start'])
inp['date'] = inp['Start'].dt.date

def hms_to_sec(t):
    """Convertit 'H:MM:SS' en secondes."""
    try:
        h, m, s = str(t).split(':')
        return int(h) * 3600 + int(m) * 60 + float(s)
    except Exception:
        return np.nan

inp['lying_sec'] = inp['Lying Time'].apply(hms_to_sec)

# Agrégat journalier
daily = inp.groupby(['Cow', 'date']).agg(
    steps_day=('Steps', 'sum'),
    lying_hours_day=('lying_sec', lambda x: x.sum() / 3600),
    lying_bouts_day=('Transitions Down', 'sum'),
    mi_day=('Motion Index', 'sum'),
).reset_index()

# Moyenne par vache + variabilité
feats = daily.groupby('Cow').agg(
    steps_per_day=('steps_day', 'mean'),
    steps_day_std=('steps_day', 'std'),
    lying_hours_per_day=('lying_hours_day', 'mean'),
    lying_bouts_per_day=('lying_bouts_day', 'mean'),
    mi_per_day=('mi_day', 'mean'),
    n_days=('date', 'nunique'),
).reset_index()

print(f"Features calculées pour {len(feats)} vaches")
feats.round(1).head(20)

Features calculées pour 17 vaches


,Cow,steps_per_day,steps_day_std,lying_hours_per_day,lying_bouts_per_day,mi_per_day,n_days
0,2047,984.0,566.8,16.5,158.5,2612.7,66
1,2056,643.5,334.0,12.8,64.5,2516.9,92
2,2063,532.0,166.5,16.1,126.7,1598.4,92
3,2069,791.6,335.0,15.1,65.2,1987.1,92
4,2081,616.5,312.1,15.0,43.9,2213.2,92
5,2083,788.2,239.0,13.1,114.6,1852.6,8
6,3437,1033.1,441.8,14.6,41.9,13549.9,86
7,3443,579.7,305.8,16.9,97.1,2700.4,86
8,5221,397.5,191.9,14.1,32.5,904.7,92
9,5246,570.1,423.4,11.7,61.7,1670.4,85


## 3. Test de séparabilité univarié (le cœur du diagnostic)

Pour chaque feature, on teste avec un Mann-Whitney U si la distribution diffère entre boiteuses et saines.
Un **p < 0.05** indiquerait une feature discriminante.

In [4]:
m = feats.merge(sls_agg, on='Cow', how='inner')
print(f"Vaches comparables (IceTag + SLS synchrone) : {len(m)}")
print(f"  boiteuses : {m['boiteuse'].sum()} | saines/légères : {(m['boiteuse']==0).sum()}\n")

feature_cols = ['steps_per_day', 'steps_day_std', 'lying_hours_per_day',
                'lying_bouts_per_day', 'mi_per_day']
labels = {
    'steps_per_day': 'Pas / jour',
    'steps_day_std': 'Variabilité des pas',
    'lying_hours_per_day': 'Heures couché / jour',
    'lying_bouts_per_day': 'Nb de couchers / jour',
    'mi_per_day': 'Motion Index / jour',
}

rows = []
for col in feature_cols:
    lame = m[m['boiteuse'] == 1][col].dropna()
    sain = m[m['boiteuse'] == 0][col].dropna()
    stat, p = mannwhitneyu(lame, sain, alternative='two-sided')
    rows.append({
        'feature': labels[col],
        'moy_boiteuses': round(lame.mean(), 2),
        'moy_saines': round(sain.mean(), 2),
        'p_value': round(p, 4),
        'significatif_5pct': 'OUI' if p < 0.05 else 'non',
    })

result = pd.DataFrame(rows)
result.to_csv(OUT / 'separabilite_univariee.csv', index=False)
print(result.to_string(index=False))
print(f"\nFeatures significatives (p<0.05) : {(result['p_value'] < 0.05).sum()} / {len(result)}")

Vaches comparables (IceTag + SLS synchrone) : 16
  boiteuses : 5 | saines/légères : 11

              feature  moy_boiteuses  moy_saines  p_value significatif_5pct
           Pas / jour         664.60      671.74   1.0000               non
  Variabilité des pas         364.56      296.05   0.3773               non
 Heures couché / jour          14.87       14.54   0.5833               non
Nb de couchers / jour          83.00       63.46   0.2674               non
  Motion Index / jour        1975.06     2993.95   1.0000               non

Features significatives (p<0.05) : 0 / 5


## 4. Visualisation : distributions par groupe

Si les boîtes se chevauchent complètement, il n'y a pas de séparation possible.

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(feature_cols), figsize=(4 * len(feature_cols), 4))
for ax, col in zip(axes, feature_cols):
    data = [m[m['boiteuse'] == 0][col].dropna(), m[m['boiteuse'] == 1][col].dropna()]
    ax.boxplot(data, labels=['Saines', 'Boiteuses'])
    # points individuels
    for i, d in enumerate(data, start=1):
        ax.scatter(np.random.normal(i, 0.05, len(d)), d, alpha=0.6, s=30)
    ax.set_title(labels[col], fontsize=10)
fig.suptitle('Winter 2019 — distributions IceTag par statut clinique SLS', fontsize=13)
fig.tight_layout()
fig.savefig(OUT / 'boxplots_separabilite.png', dpi=120, bbox_inches='tight')
print('Figure sauvegardée :', OUT / 'boxplots_separabilite.png')
plt.show()

Figure sauvegardée : /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/diagnostic_separabilite/boxplots_separabilite.png


## 5. Plafond multivarié : un classifieur peut-il faire mieux ?

Une feature seule peut échouer alors qu'une **combinaison** réussit. On entraîne un classifieur
(Random Forest) avec validation croisée Leave-One-Out (adaptée au petit échantillon) pour mesurer
le **plafond de performance atteignable** sur ces données.

Référence : un AUC de 0.5 = hasard pur (aucun signal). Un AUC proche de 1.0 = signal fort.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

X = m[feature_cols].fillna(m[feature_cols].median())
y = m['boiteuse'].values

clf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
loo = LeaveOneOut()

# Probabilités prédites en LOO
proba = cross_val_predict(clf, X, y, cv=loo, method='predict_proba')[:, 1]
pred = (proba >= 0.5).astype(int)

auc = roc_auc_score(y, proba)
bacc = balanced_accuracy_score(y, pred)
print(f"AUC (Leave-One-Out)         : {auc:.3f}   (0.5 = hasard)")
print(f"Balanced accuracy           : {bacc:.3f}   (0.5 = hasard)")
print()
if auc < 0.6:
    print("=> AUC proche du hasard : MÊME une combinaison de features ne sépare pas les groupes.")
    print("   Le signal de boiterie n'est pas présent dans les features IceTag de cette cohorte.")
elif auc < 0.75:
    print("=> Signal faible/ambigu. Une recalibration McGill aurait un intérêt limité.")
else:
    print("=> Signal exploitable détecté : une recalibration McGill du pipeline est justifiée.")

pd.DataFrame({'Cow': m['Cow'], 'SLS_total': m['SLS_total'], 'boiteuse': y,
              'proba_predite': proba.round(3)}).to_csv(
    OUT / 'classifieur_loo_predictions.csv', index=False)

AUC (Leave-One-Out)         : 0.236   (0.5 = hasard)
Balanced accuracy           : 0.409   (0.5 = hasard)

=> AUC proche du hasard : MÊME une combinaison de features ne sépare pas les groupes.
   Le signal de boiterie n'est pas présent dans les features IceTag de cette cohorte.


## 6. Rappel : alertes du pipeline gelé vs SLS

On confirme que les notifications du pipeline (tel quel, paramètres du mémoire) ne corrèlent pas avec le SLS.

In [7]:
summ = pd.read_csv(PIPELINE_SUMMARY)
summ['Cow'] = summ['Cow'].astype(str)
ms = summ.merge(sls_agg, on='Cow', how='inner')

rho, p = spearmanr(ms['SLS_total'], ms['lameness_notifs'])
lame_n = ms[ms['boiteuse'] == 1]['lameness_notifs']
sain_n = ms[ms['boiteuse'] == 0]['lameness_notifs']
stat, pmw = mannwhitneyu(lame_n, sain_n, alternative='two-sided')

print(f"Notifications moyennes — boiteuses : {lame_n.mean():.2f} | saines : {sain_n.mean():.2f}")
print(f"Mann-Whitney (notifs) p = {pmw:.4f}")
print(f"Corrélation Spearman (SLS vs notifs) : rho = {rho:.3f}, p = {p:.4f}")
print("\n=> Cohérent avec le diagnostic : pas de relation entre alertes et boiterie clinique.")

Notifications moyennes — boiteuses : 8.40 | saines : 9.64
Mann-Whitney (notifs) p = 0.6487
Corrélation Spearman (SLS vs notifs) : rho = 0.033, p = 0.9037

=> Cohérent avec le diagnostic : pas de relation entre alertes et boiterie clinique.


## 7. Conclusion du diagnostic

On rassemble les trois verdicts dans une note exploitable.

In [8]:
n_sig = int((result['p_value'] < 0.05).sum())
verdict = []
verdict.append('# Diagnostic de séparabilité — Winter 2019 (labels SLS synchrones)\n')
verdict.append(f'Vaches analysées : {len(m)} (boiteuses SLS>=2 : {int(m["boiteuse"].sum())}, '
               f'saines/légères : {int((m["boiteuse"]==0).sum())})\n')
verdict.append('## 1. Séparabilité univariée')
verdict.append(f'Features IceTag testées : {len(result)} | significatives (p<0.05) : {n_sig}\n')
verdict.append(result.to_string(index=False))
verdict.append(f'\n## 2. Plafond multivarié (Random Forest, Leave-One-Out)')
verdict.append(f'AUC = {auc:.3f} | balanced accuracy = {bacc:.3f} (0.5 = hasard)\n')
verdict.append('## 3. Pipeline gelé vs SLS')
verdict.append(f'Spearman rho = {rho:.3f} (p={p:.3f}) | Mann-Whitney notifs p = {pmw:.3f}\n')
verdict.append('## Conclusion')
if n_sig == 0 and auc < 0.6:
    verdict.append(
        "Aucune feature IceTag ne sépare les vaches boiteuses des saines, et un classifieur "
        "multivarié ne fait pas mieux que le hasard. Le signal de boiterie n'est PAS présent "
        "dans les données IceTag de cette cohorte d'étude d'exercice. \n\n"
        "=> L'absence de détection n'est imputable NI au pipeline NI au travail du mémoire : "
        "c'est une limite des données/de la cohorte. On ne peut pas détecter un signal absent "
        "de l'entrée. Le mémoire reste valide (validé par injection sur ses propres données).")
else:
    verdict.append(
        "Un signal partiel existe. Une recalibration McGill-spécifique du pipeline "
        "(notebook 07) est justifiée pour en mesurer la portée.")

note = '\n'.join(verdict)
(OUT / 'diagnostic_conclusion.md').write_text(note, encoding='utf-8')
print(note)
print('\nNote sauvegardée :', OUT / 'diagnostic_conclusion.md')

# Diagnostic de séparabilité — Winter 2019 (labels SLS synchrones)

Vaches analysées : 16 (boiteuses SLS>=2 : 5, saines/légères : 11)

## 1. Séparabilité univariée
Features IceTag testées : 5 | significatives (p<0.05) : 0

              feature  moy_boiteuses  moy_saines  p_value significatif_5pct
           Pas / jour         664.60      671.74   1.0000               non
  Variabilité des pas         364.56      296.05   0.3773               non
 Heures couché / jour          14.87       14.54   0.5833               non
Nb de couchers / jour          83.00       63.46   0.2674               non
  Motion Index / jour        1975.06     2993.95   1.0000               non

## 2. Plafond multivarié (Random Forest, Leave-One-Out)
AUC = 0.236 | balanced accuracy = 0.409 (0.5 = hasard)

## 3. Pipeline gelé vs SLS
Spearman rho = 0.033 (p=0.904) | Mann-Whitney notifs p = 0.649

## Conclusion
Aucune feature IceTag ne sépare les vaches boiteuses des saines, et un classifieur multivarié ne fait p